# FDIA Alarm Defense — Demo Experiment (TEP Fault 5)

Companion notebook for **"A Novel False Data Injection Attack Threat Model Targeting Industrial
Fault Detectors and Corresponding Countermeasures"** (IEEE Transactions on Industrial Informatics,
2026, Paper ID: TII-26-5321).

This notebook walks through the full pipeline on a single scenario (TEP Fault 5 / IDV5):

1. Load the Tennessee Eastman Process (TEP) dataset
2. Train the DAE-PCA fault detector
3. Compute the T² / SPE detection thresholds
4. Compute the baseline False Alarm Rate (FAR), before any attack
5. Generate the chattering False Data Injection Attack (FDIA)
6. Compare FAR before vs. after the attack
7. Plot the SPE trajectory (baseline vs. attacked)
8. Apply the delay-timer countermeasure and measure the FAR it recovers

**Prerequisite:** download the TEP dataset and place it under `data/TEdata/` as described in
[`data/README_data.md`](../data/README_data.md) (`X.mat`, `Xv.mat`, `Xt1.mat` ... `Xt21.mat`).

This notebook trains with a reduced epoch budget so it runs quickly; see the note in the
training section for how to reproduce the paper's full setting.


## 0. Setup and imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import torch as t
import matplotlib.pyplot as plt

# Jupyter's default cwd is the notebook's own directory (notebooks/); resolve the repo
# layout from there so this works regardless of how the notebook was launched.
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
SRC_DIR = REPO_ROOT / "src"
DATA_DIR = REPO_ROOT / "data"

sys.path.insert(0, str(SRC_DIR))

np.random.seed(42)
t.manual_seed(42)

print(f"Repo root: {REPO_ROOT}")
print(f"Source dir: {SRC_DIR}")


In [ ]:
from dataset import dataset
from model import AE
import config as opt
import thresholds
from utils import BICcom, FDR, FAR
from exp_harness import (
    ChatteringConfig,
    run_vacf_fault_free_experiment,
    apply_countermeasure_block,
    far_from_alarms,
)

# Training / evaluation helpers (train, val, thre, tes, eval_far_fdr, build_spe_fn,
# build_apply_delta_step) already implemented in main.py -- reused here instead of
# duplicating the DAE-PCA training loop. main.py only runs its pipeline under
# `if __name__ == "__main__"`, so importing it is safe.
from main import _to_tensor, train, val, thre, tes, eval_far_fdr, build_spe_fn, build_apply_delta_step

# attack_generator.py is imported later (Section 5), once config.J_SPE has been
# computed -- it does `from config import J_SPE` at import time, and config.py only
# defines J_SPE once the threshold has actually been calculated (see Section 3).


## 1. Load the TEP data

`dataset(I)` (in `src/dataset.py`) loads `X.mat` / `Xv.mat` / `Xt{I}.mat` from a path that is
relative to the current working directory (`./TEdata`). We temporarily switch into `data/`
so it resolves to `data/TEdata/`, matching the layout documented in `data/README_data.md`.

- IDV=1 (normal operating data) is used to train the detector.
- IDV=5 (TEP Fault 5) is the faulty test trajectory used throughout the rest of this demo.


In [ ]:
FAULT_ID = 5  # TEP Fault 5 (IDV5)


def load_tep_split(idv: int):
    cwd = os.getcwd()
    try:
        os.chdir(DATA_DIR)
        return dataset(idv)
    finally:
        os.chdir(cwd)


train_loader, val_loader, _ = load_tep_split(1)        # IDV=1: normal data, for training
_, _, test_loader = load_tep_split(FAULT_ID)            # IDV=5: faulty trajectory, for evaluation

train_x = _to_tensor(train_loader)
val_x = _to_tensor(val_loader)
test_x = _to_tensor(test_loader)

print(f"train_x: {tuple(train_x.shape)}  val_x: {tuple(val_x.shape)}  test_x (IDV{FAULT_ID}): {tuple(test_x.shape)}")


## 2. Train the DAE-PCA model

The paper trains for `opt.max_epoch` (20000) epochs. For a quick interactive demo we use a
much smaller `DEMO_EPOCHS` budget by default -- set `DEMO_EPOCHS = opt.max_epoch` to
reproduce the paper's full training run.


In [ ]:
DEMO_EPOCHS = 2000  # reduce for a faster demo, or set to opt.max_epoch (20000) for full reproduction

Ae = AE(opt)

best_path = str(SRC_DIR / "model_state" / "best.pth.tar")  # mirrors opt.model_state_path from config.py
os.makedirs(os.path.dirname(best_path), exist_ok=True)

min_loss = float("inf")
best_epoch = -1

for epoch in range(DEMO_EPOCHS):
    Ae, train_loss, l1, l2 = train(Ae, train_x, epoch)
    val_loss = val(Ae, val_x)

    if val_loss < min_loss:
        min_loss = val_loss
        best_epoch = epoch
        t.save(Ae.state_dict(), best_path)

    if epoch % 200 == 0:
        print(f"Epoch [{epoch + 1}/{DEMO_EPOCHS}] | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

print(f"Best checkpoint: epoch {best_epoch}, val_loss={min_loss:.6f} -> {best_path}")


## 3. Compute the detection thresholds (T², SPE)

`thre()` (from `main.py`) computes the T² and SPE statistics on the normal training data.
We then use `thresholds.calculate_threshold` (KDE-based, `src/thresholds.py`) -- one of the
modules this demo is built around -- to turn those statistics into the `J_T2` / `J_SPE`
detection thresholds at a 99% confidence level.


In [ ]:
T2c, SPEc, cov_T, T_train = thre(Ae, best_path, train_x)

J_T2 = thresholds.calculate_threshold(T2c, alpha=0.99)
J_SPE = thresholds.calculate_threshold(SPEc, alpha=0.99)

print(f"Detection thresholds -> J_T2={J_T2:.4f}, J_SPE={J_SPE:.4f}")

# Make J_SPE available on the config module so attack_generator's
# `from config import J_SPE` succeeds (see the Setup note above).
opt.J_SPE = J_SPE

from attack_generator import generate_chattering_attack


## 4. Baseline FAR/FDR on TEP Fault 5 (before any attack)

`tes()` runs the trained detector over the IDV=5 trajectory to get per-sample T², SPE and BIC
statistics. `eval_far_fdr()` then computes FAR/FDR over the fault-free window `[0, f)` vs. the
faulty region `[f, N)`, where `f = opt.f`. We cross-check `FAR_SPE` directly with `utils.FAR`.


In [ ]:
f_win = int(opt.f)

T2, SPE, BIC = tes(best_path, test_x, cov_T, J_T2, J_SPE)
baseline_metrics = eval_far_fdr(T2, SPE, BIC, f_win, J_T2, J_SPE)

far_spe_check = FAR(SPE, J_SPE, f_win)  # utils.FAR, same quantity as baseline_metrics['FAR_SPE']

print(f"Baseline (no attack), IDV={FAULT_ID}:")
for k, v in baseline_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"  FAR_SPE (utils.FAR direct check): {far_spe_check:.4f}")


## 5. Generate the chattering FDIA attack

First, a single-sample illustration with `attack_generator.generate_chattering_attack`
(gradient-based perturbation that drives the SPE across `J_SPE`). Then, the full
fault-free-trajectory version used in the paper: `exp_harness.run_vacf_fault_free_experiment`
applies the chattering gate/perturbation defined by `ChatteringConfig` across the fault-free
window and reports FAR before vs. after in one call.


In [ ]:
sample_idx = 0
x_sample = test_x[sample_idx : sample_idx + 1]

Ae.eval()
x_attacked = generate_chattering_attack(x_sample, Ae, opt, alpha=0.5, beta=0.3, omega=10)

spe_fn = build_spe_fn(best_path)
spe_clean = spe_fn(x_sample.numpy())
spe_attacked = spe_fn(x_attacked.numpy())

print(f"Sample {sample_idx}: SPE clean={spe_clean:.4f} -> SPE attacked={spe_attacked:.4f} (J_SPE={J_SPE:.4f})")


In [ ]:
apply_delta_step = build_apply_delta_step()

X_fault_free = test_x[:f_win].numpy()  # fault-free portion of the IDV=5 trajectory (before fault onset)

cfg = ChatteringConfig(k=3, eps_max=0.05, gate_mode="bernoulli", rho=0.4)
rng = np.random.default_rng(42)

vacf_result = run_vacf_fault_free_experiment(
    X=X_fault_free,
    spe_fn=spe_fn,
    apply_delta_step=apply_delta_step,
    J_SPE=J_SPE,
    cfg=cfg,
    rng=rng,
)

print("Chattering FDIA attack applied over the fault-free window.")


## 6. FAR before vs. after the attack

In [ ]:
print(f"FAR_base   (no attack):    {vacf_result['FAR_base']:.4f}")
print(f"FAR_attack (chattering):    {vacf_result['FAR_attack_emp']:.4f}")
print(f"Delta FAR:                  {vacf_result['delta_FAR_emp']:+.4f}")


## 7. Plot SPE comparison (baseline vs. attacked)

In [ ]:
SPE0 = vacf_result["SPE0"]
SPE1 = vacf_result["SPE1"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(SPE0, label="SPE (baseline, no attack)", alpha=0.8)
ax.plot(SPE1, label="SPE (under chattering FDIA)", alpha=0.8)
ax.axhline(J_SPE, color="black", linestyle="--", label=r"Threshold $J_{SPE}$")
ax.set_xlabel("Sample index")
ax.set_ylabel("SPE")
ax.set_title(f"SPE comparison -- baseline vs. attacked (IDV={FAULT_ID}, fault-free window)")
ax.legend()
fig.tight_layout()
plt.show()


## 8. Apply the delay-timer countermeasure

`exp_harness.apply_countermeasure_block` implements the paper's countermeasure: a
chattering-aware adaptive threshold plus a consecutive-`Kc` delay-timer alarm policy. We run
it on the attacked SPE series and compare its defended FAR against the raw (undefended)
attacked FAR via `far_from_alarms`.


In [ ]:
sigma_spe_x0 = float(np.std(SPEc, ddof=1))  # SPE std on fault-free training data

alarms_def, alarms_adapted_raw, J_eff_series, nu_series = apply_countermeasure_block(
    spe=SPE1,
    J=J_SPE,
    sigma_spe_x0=sigma_spe_x0,
    Kc=2,
    L=20,
    nu_max=6,
    delta_factor=0.5,
)

far_attack_raw = far_from_alarms(vacf_result["alarms1"])
far_attack_defended = far_from_alarms(alarms_def)

print(f"FAR under attack, undefended:                 {far_attack_raw:.4f}")
print(f"FAR under attack, defended (delay-timer Kc=2): {far_attack_defended:.4f}")
print(f"FAR reduction from the countermeasure:         {far_attack_raw - far_attack_defended:+.4f}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(vacf_result["alarms1"], label="Raw alarms (attacked, undefended)", alpha=0.6)
ax.plot(alarms_def, label="Defended alarms (delay-timer, Kc=2)", alpha=0.8)
ax.set_xlabel("Sample index")
ax.set_ylabel("Alarm (0/1)")
ax.set_title("Effect of the delay-timer countermeasure on alarm output")
ax.legend()
fig.tight_layout()
plt.show()


## Summary

- The DAE-PCA detector was trained on normal TEP operating data and thresholded at the 99%
  confidence level (`J_T2`, `J_SPE`).
- The chattering FDIA attack (Section 5) drives the SPE statistic to repeatedly cross
  `J_SPE`, raising the empirical FAR well above baseline (Section 6).
- The delay-timer countermeasure (Section 8) filters out the attack-induced chattering
  alarms, recovering a large share of the FAR lift without a full retrain of the detector.

See `figs/` for the corresponding publication-quality figures across all 21 TEP fault
scenarios, and `src/make_figs.py` for how they were generated.
